<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_ia/blob/main/GANS_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StyleGAN3 : de l'image générée à l'animation

Dans ce notebook, on utilise **StyleGAN3** pour générer une petite animation.

## Idée principale
StyleGAN2 permet déjà de générer de très bonnes images fixes.  
StyleGAN3 cherche à rendre la génération **plus cohérente quand l'image évolue**.

## Pourquoi c'est intéressant ?
Si l'on veut produire une séquence d'images, ou une vidéo, il ne suffit pas que chaque image soit belle séparément.  
Il faut aussi que les détails restent stables et se déplacent de manière naturelle.

## Objectif du notebook
Nous allons :

1. charger un modèle préentraîné StyleGAN3 ;
2. générer deux visages différents ;
3. interpoler entre leurs vecteurs latents ;
4. transformer cette interpolation en GIF ou en vidéo.

## Ce qu'on veut comprendre
- ce qu'est une interpolation latente ;
- pourquoi StyleGAN3 est intéressant pour l'animation ;
- en quoi cela prépare aux modèles vidéo plus récents.

## Rappel : la logique générale

Dans ce type de modèle, on peut résumer le processus ainsi :

**seed -> vecteur latent -> image**

- le **seed** est un nombre entier ;
- ce seed permet de générer un **vecteur latent** ;
- le modèle transforme ce vecteur en image.

Si l'on fait varier progressivement le vecteur latent, on peut produire une série d'images qui évoluent progressivement.

In [ ]:
!nvidia-smi || true

import sys
import platform
import torch

print("Python :", sys.version)
print("Plateforme :", platform.platform())
print("Torch version :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

## Installation minimale

On garde l'environnement PyTorch de Colab et on installe seulement ce qui est utile :
- affichage d'images ;
- création de GIF/vidéos ;
- clonage du dépôt officiel.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y build-essential ninja-build ffmpeg
!pip -q install numpy pillow matplotlib imageio imageio-ffmpeg

## Récupérer le code officiel StyleGAN3

On clone ici le dépôt officiel NVIDIA de StyleGAN3.

Ce dépôt contient notamment :
- le code du générateur ;
- les scripts de génération ;
- les fonctions nécessaires pour produire des images à partir de vecteurs latents.

In [ ]:
!rm -rf /content/stylegan3
!git clone https://github.com/NVlabs/stylegan3.git /content/stylegan3
%cd /content/stylegan3

## Quel modèle choisir ?

StyleGAN3 existe en plusieurs variantes.

Deux familles importantes :

- **stylegan3-t** : bon comportement en translation ;
- **stylegan3-r** : bon comportement en translation et en rotation.

Pour un premier notebook, on choisit un modèle de visages préentraîné.

## Idée pédagogique
La logique d'usage ressemble à StyleGAN2 :
- on choisit un modèle ;
- on choisit des seeds ;
- on génère des images.

La différence n'est pas surtout dans l'interface, mais dans la manière dont le modèle gère les détails visuels quand l'image change.

In [ ]:
NETWORK_URL = "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/files/stylegan3-r-ffhq-1024x1024.pkl"
print("Modèle choisi :", NETWORK_URL)

## Générer deux images de départ

On va commencer par produire deux visages différents.

### Pourquoi deux ?
Parce qu'ensuite, nous allons faire une transition progressive entre eux.

### Important
On utilise deux **seeds différents**.  
Chaque seed correspond à un point différent dans l'espace latent.

In [ ]:
import os
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

import dnnlib
import legacy

from torch_utils.ops import bias_act, upfirdn2d, filtered_lrelu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_original_bias_act = bias_act.bias_act
_original_upfirdn2d = upfirdn2d.upfirdn2d
_original_filtered_lrelu = filtered_lrelu.filtered_lrelu

def bias_act_ref(*args, **kwargs):
    kwargs["impl"] = "ref"
    return _original_bias_act(*args, **kwargs)

def upfirdn2d_ref(*args, **kwargs):
    kwargs["impl"] = "ref"
    return _original_upfirdn2d(*args, **kwargs)

def filtered_lrelu_ref(*args, **kwargs):
    kwargs["impl"] = "ref"
    return _original_filtered_lrelu(*args, **kwargs)

bias_act.bias_act = bias_act_ref
upfirdn2d.upfirdn2d = upfirdn2d_ref
filtered_lrelu.filtered_lrelu = filtered_lrelu_ref

print("Ops forcées en mode ref.")

In [ ]:
seed_a = 10
seed_b = 200
truncation = 0.7

with dnnlib.util.open_url(NETWORK_URL) as f:
    G = legacy.load_network_pkl(f)["G_ema"].to(device)

print("Générateur chargé.")
print("z_dim =", G.z_dim)
print("c_dim =", G.c_dim)

labels = torch.zeros([1, G.c_dim], device=device)

def generate_from_seed(seed, outpath):
    rng = np.random.RandomState(seed)
    z = torch.from_numpy(rng.randn(1, G.z_dim)).to(device=device, dtype=torch.float32)

    with torch.no_grad():
        img = G(z, labels, truncation_psi=truncation, noise_mode="const")
        img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
        img_pil = Image.fromarray(img[0].cpu().numpy(), "RGB")
        img_pil.save(outpath)

os.makedirs("/content/stylegan3_outputs_start", exist_ok=True)
generate_from_seed(seed_a, "/content/stylegan3_outputs_start/seed0010.png")
generate_from_seed(seed_b, "/content/stylegan3_outputs_start/seed0200.png")

print("Deux images générées.")

In [ ]:
img1 = Image.open("/content/stylegan3_outputs_start/seed0010.png")
img2 = Image.open("/content/stylegan3_outputs_start/seed0200.png")

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(img1)
plt.title("Seed 10")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(img2)
plt.title("Seed 200")
plt.axis("off")

plt.tight_layout()
plt.show()

## Interpoler entre deux points latents

Interpoler, ici, veut dire :

> passer progressivement d'un vecteur latent à un autre

Autrement dit, au lieu de générer seulement deux images séparées, on va construire
des images intermédiaires entre les deux.

## Pourquoi c'est intéressant ?
Cela permet de visualiser l'espace latent comme un espace continu :
- on ne saute pas brutalement d'une image à une autre ;
- on traverse progressivement des états intermédiaires.

## Ce qu'on va faire
Nous allons :
1. générer un vecteur latent pour le seed 10 ;
2. générer un vecteur latent pour le seed 200 ;
3. calculer des points intermédiaires ;
4. transformer chacun de ces points en image.

In [ ]:
import os
import numpy as np
import torch
import pickle
from PIL import Image
import dnnlib
import legacy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with dnnlib.util.open_url(NETWORK_URL) as f:
    G = legacy.load_network_pkl(f)["G_ema"].to(device)

print("Générateur chargé.")
print("Dimension z :", G.z_dim)
print("Dimension c :", G.c_dim)

## Créer les vecteurs latents à partir des seeds

Le seed est un entier, mais le générateur a besoin d'un vecteur latent.

Nous allons donc :
- fixer un seed ;
- produire un vecteur aléatoire `z` ;
- faire cela pour deux seeds différents.

In [ ]:
seed_a = 10
seed_b = 200
truncation = 0.7

rng_a = np.random.RandomState(seed_a)
rng_b = np.random.RandomState(seed_b)

z_a = torch.from_numpy(rng_a.randn(1, G.z_dim)).to(device=device, dtype=torch.float32)
z_b = torch.from_numpy(rng_b.randn(1, G.z_dim)).to(device=device, dtype=torch.float32)

print("z_a shape :", z_a.shape)
print("z_b shape :", z_b.shape)

## Générer des images intermédiaires

On va fabriquer plusieurs étapes entre `z_a` et `z_b`.

### Idée simple
- au début : image proche de `z_a`
- à la fin : image proche de `z_b`
- entre les deux : états intermédiaires

Cette transition progressive est ce qu'on appelle une **interpolation latente**.

In [ ]:
outdir = "/content/interpolation_frames"
os.makedirs(outdir, exist_ok=True)

num_steps = 24

labels = torch.zeros([1, G.c_dim], device=device)

for i, alpha in enumerate(np.linspace(0, 1, num_steps)):
    z = (1 - alpha) * z_a + alpha * z_b

    img = G(z, labels, truncation_psi=truncation, noise_mode="const")
    img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8)
    img_pil = Image.fromarray(img[0].cpu().numpy(), "RGB")
    img_pil.save(f"{outdir}/frame_{i:03d}.png")

print("Frames enregistrées dans :", outdir)

In [ ]:
import glob

frames = sorted(glob.glob("/content/interpolation_frames/*.png"))

plt.figure(figsize=(16,8))
for i, fp in enumerate([frames[0], frames[6], frames[12], frames[18], frames[23]], start=1):
    plt.subplot(1,5,i)
    plt.imshow(Image.open(fp))
    plt.title(os.path.basename(fp))
    plt.axis("off")
plt.tight_layout()
plt.show()

## Transformer les images en GIF

Nous avons maintenant une série d'images.  
Nous allons les assembler en GIF.

## Ce que cela montre
Une vidéo générée n'est souvent, au départ, qu'une suite d'images produites de façon cohérente.

In [ ]:
import imageio.v2 as imageio

gif_path = "/content/stylegan3_interpolation.gif"

images = [imageio.imread(fp) for fp in frames]
imageio.mimsave(gif_path, images, duration=0.10)

print("GIF créé :", gif_path)

In [ ]:
from IPython.display import Image as IPyImage, display

display(IPyImage(filename="/content/stylegan3_interpolation.gif"))

## Transformer les images en vidéo MP4

Le GIF est pratique pour visualiser rapidement.  
Mais une vidéo MP4 est souvent plus utile pour conserver une meilleure qualité.

In [ ]:
!ffmpeg -y -framerate 12 -i /content/interpolation_frames/frame_%03d.png \
    -pix_fmt yuv420p /content/stylegan3_interpolation.mp4

In [ ]:
from IPython.display import Video, display

display(Video("/content/stylegan3_interpolation.mp4", embed=True, width=640))

## Qu'est-ce que StyleGAN3 apporte ici ?

Dans ce notebook, la logique générale est proche de StyleGAN2 :
- un modèle préentraîné ;
- des seeds ;
- un espace latent ;
- des images générées.

## La différence principale
StyleGAN3 cherche à rendre les détails visuels plus cohérents lorsque l'image change.

Cela devient particulièrement intéressant pour :
- les interpolations ;
- les animations ;
- les séquences vidéo générées.

## Idée simple
StyleGAN2 est très bon pour générer une image fixe.  
StyleGAN3 est plus intéressant quand on veut faire évoluer cette image de façon fluide.

## Ce qu'il faut retenir

Dans ce notebook, on a vu que :

- un seed permet de produire un vecteur latent ;
- deux seeds différents donnent deux points différents dans l'espace latent ;
- on peut interpoler entre ces deux points ;
- cette interpolation produit une suite d'images cohérentes ;
- une vidéo générée peut donc être comprise comme une succession d'images produites de manière continue.

## Formule simple
On peut résumer ainsi :

**deux points latents + interpolation -> séquence d'images -> GIF ou vidéo**